# Training curves

Plots columns of an Ultralytics `results.csv` (works on running, stopped, and finished runs).

- **Same range for every curve (default):** each series is min-max scaled to 0 to 1 on one shared axis, so a loss and F2 are equally readable. The legend keeps each series' real range. Shapes and trends compare across series; heights do not mean anything across series.
- **Scale ignores first K epochs (default 1):** warmup epochs can be huge outliers (c01's epoch-1 val cls loss is 13.7) and would flatten a series; they are left out of its 0 to 1 scale and may fall off the plot.
- **Separate panels:** real units, one panel per quantity (train and val of the same loss share a panel).
- **Color = quantity, line style = split:** solid for train and metrics, dashed for val losses.
- **F2 moving average:** trailing mean of the last N epochs (fewer at the start), scaled with raw F2's own range so the two stay comparable.

Re-run **Reload** to pick up new epochs while a run is training.

In [1]:
import csv
from pathlib import Path

import ipywidgets as w
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())

# Fixed color per quantity (reference categorical palette, light mode), so a column keeps
# its color whatever else is selected. Quantities past the 8 hues get neutral gray + their own dash.
SERIES_STYLE = {
    "box_loss":     ("#2a78d6", None),
    "cls_loss":     ("#eb6834", None),
    "l1_loss":      ("#1baf7a", None),
    "F2":           ("#6250d6", None),
    "recall(B)":    ("#008300", None),
    "precision(B)": ("#e87ba4", None),
    "mAP50(B)":     ("#eda100", None),
    "mAP50-95(B)":  ("#e34948", None),
    "F2_conf":      ("#898781", (0, (1, 1.5))),
    "time":         ("#898781", (0, (4, 2, 1, 2))),
}
LR_STYLE = ("#898781", (0, (6, 2)))
SURFACE, INK, INK_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e0"


def load_results(path):
    with open(path, newline="") as f:
        rows = [{k.strip(): v for k, v in r.items()} for r in csv.DictReader(f)]
    return {k: np.array([float(r[k]) for r in rows]) for k in rows[0]} if rows else {}


def quantity(col):
    return col.split("/", 1)[-1]


def style(col):
    if col.startswith("lr/"):
        color, dash = LR_STYLE
    else:
        color, dash = SERIES_STYLE.get(quantity(col), ("#898781", (0, (2, 2))))
    if col.startswith("val/"):
        dash = (0, (5, 2.5))
    return color, (dash or "solid")


def trailing_mean(y, n):
    c = np.cumsum(np.insert(y, 0, 0.0))
    idx = np.arange(1, len(y) + 1)
    lo = np.maximum(idx - n, 0)
    return (c[idx] - c[lo]) / (idx - lo)


def scale01(y, lo, hi):
    return (y - lo) / (hi - lo) if hi > lo else np.full_like(y, 0.5)


def plot_results(path, columns, normalize=True, separate=False, ma_window=None, mark_best=True, skip=1):
    """skip: in same-range mode, epochs excluded when computing each series' 0-1 scale (warmup outliers);
    their points can fall off the plot."""
    data = load_results(path)
    if not data:
        print(f"{path}: no epochs yet"); return
    ep = data["epoch"]
    has_f2 = "metrics/F2" in data
    series = [(c, data[c], c, None) for c in columns if c in data]  # (label, y, style key, scale source)
    if ma_window and has_f2:
        series.append((f"F2, mean of last {ma_window}", trailing_mean(data["metrics/F2"], ma_window), "metrics/F2", "metrics/F2"))
    if not series:
        print("Select at least one column."); return
    best = int(np.argmax(data["metrics/F2"])) if has_f2 else None

    plt.rcParams.update({"axes.edgecolor": GRID, "axes.labelcolor": INK_2, "xtick.color": INK_2, "ytick.color": INK_2,
                         "text.color": INK, "font.size": 10})
    groups = {}
    for s in series:
        groups.setdefault(quantity(s[2]) if separate else "all", []).append(s)
    n = len(groups)
    ncols = 1 if n == 1 else 2
    nrows = -(-n // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(11 if ncols == 2 else (12 if normalize and not separate else 9), 3.6 * nrows if separate else 5),
                             sharex=True, squeeze=False, facecolor=SURFACE)
    for ax in axes.flat[n:]:
        ax.set_visible(False)

    for ax, (gname, group) in zip(axes.flat, groups.items()):
        ax.set_facecolor(SURFACE)
        for label, y, key, scale_src in group:
            color, dash = style(key)
            src = data[scale_src] if scale_src else y
            ref = src[skip:] if (normalize and not separate and len(src) > skip) else src
            lo, hi = float(np.min(ref)), float(np.max(ref))
            is_ma = label.startswith("F2, mean")
            yy = scale01(y, lo, hi) if (normalize and not separate) else y
            legend = label
            if normalize and not separate:
                legend += "  (scaled with F2)" if is_ma else f"  ({lo:.4g} to {hi:.4g})"
            ax.plot(ep, yy, color=color, linestyle=dash, linewidth=3 if is_ma else 2,
                    alpha=1.0 if (is_ma or not ma_window or key != "metrics/F2") else 0.45, label=legend)
        if mark_best and best is not None:
            ax.axvline(ep[best], color=INK_2, linewidth=1, linestyle=(0, (2, 3)))
        ax.grid(True, color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        if separate:
            ax.set_title(gname, color=INK, fontsize=10, loc="left")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        if normalize and not separate:
            ax.legend(frameon=False, fontsize=8.5, loc="upper left", bbox_to_anchor=(1.01, 1))
        else:
            ax.legend(frameon=False, fontsize=8.5, loc="best")
    for ax in axes[-1]:
        ax.set_xlabel("epoch")
    if normalize and not separate:
        axes[0, 0].set_ylabel("scaled per series (0 = its min, 1 = its max)" + (f", epochs > {skip}" if skip else ""))
        axes[0, 0].set_ylim(-0.05, 1.05)

    run = Path(path).parent
    title = f"{run.parent.name} / {run.name}  ({len(ep)} epochs)"
    if best is not None:
        title += f"   best F2 {data['metrics/F2'][best]:.3f} at epoch {int(ep[best])}" + (" (dotted line)" if mark_best else "")
    fig.suptitle(title, color=INK, fontsize=11, x=0.01, ha="left")
    fig.tight_layout()
    plt.show()

In [2]:
DEFAULT = ROOT / "model_dev/c02_yolo26s_1280_seq20val_pos/seed0/results.csv"

path_box = w.Text(value=str(DEFAULT), description="results.csv", layout=w.Layout(width="95%"),
                  style={"description_width": "initial"})
reload_btn = w.Button(description="Reload", icon="refresh")
normalize = w.Checkbox(value=True, description="Same range for every curve")
separate = w.Checkbox(value=False, description="Separate panels (real units)")
ma_on = w.Checkbox(value=True, description="F2 moving average")
ma_n = w.IntSlider(value=5, min=2, max=20, description="last N epochs", style={"description_width": "initial"})
mark_best = w.Checkbox(value=True, description="Mark best F2 epoch")
skip = w.IntSlider(value=1, min=0, max=10, description="scale ignores first K epochs", style={"description_width": "initial"})
preset_overfit = w.Button(description="Preset: overfitting (losses)")
preset_metrics = w.Button(description="Preset: metrics")
col_box = w.VBox()
out = w.Output()
checks = {}

def build_checks(*_):
    global checks
    try:
        cols = [c for c in load_results(path_box.value) if c != "epoch"]
    except (OSError, StopIteration, KeyError) as e:
        with out:
            out.clear_output(); print(f"cannot read {path_box.value}: {e}")
        return
    old = {c: cb.value for c, cb in checks.items()}
    default_on = {"train/box_loss", "val/box_loss", "train/cls_loss", "val/cls_loss", "metrics/F2"}
    checks = {c: w.Checkbox(value=old.get(c, c in default_on), description=c, indent=False,
                            layout=w.Layout(width="210px")) for c in cols}
    for cb in checks.values():
        cb.observe(redraw, "value")
    groups = {"train": [], "val": [], "metrics": [], "other": []}
    for c, cb in checks.items():
        groups[c.split("/")[0] if c.split("/")[0] in groups else "other"].append(cb)
    col_box.children = [w.HBox([w.Label(g, layout=w.Layout(width="70px")), w.HBox(cbs, layout=w.Layout(flex_flow="row wrap"))])
                        for g, cbs in groups.items() if cbs]
    ma_on.disabled = ma_n.disabled = "metrics/F2" not in checks
    redraw()

def redraw(*_):
    with out:
        out.clear_output(wait=True)
        plot_results(path_box.value, [c for c, cb in checks.items() if cb.value], normalize=normalize.value,
                     separate=separate.value, ma_window=ma_n.value if (ma_on.value and not ma_on.disabled) else None,
                     mark_best=mark_best.value, skip=skip.value)

def apply_preset(cols):
    for c, cb in checks.items():
        cb.unobserve(redraw, "value"); cb.value = c in cols; cb.observe(redraw, "value")
    redraw()

preset_overfit.on_click(lambda _: apply_preset({"train/box_loss", "val/box_loss", "train/cls_loss", "val/cls_loss",
                                                "train/l1_loss", "val/l1_loss"}))
preset_metrics.on_click(lambda _: apply_preset({"metrics/F2", "metrics/recall(B)", "metrics/precision(B)", "metrics/mAP50(B)"}))
reload_btn.on_click(build_checks)
path_box.on_submit(build_checks)
for wd in (normalize, separate, ma_on, ma_n, mark_best, skip):
    wd.observe(redraw, "value")

display(w.VBox([w.HBox([path_box, reload_btn]), col_box,
                w.HBox([normalize, separate, mark_best]), w.HBox([ma_on, ma_n, skip]),
                w.HBox([preset_overfit, preset_metrics]), out]))
build_checks()

/tmp/ipykernel_97925/816657840.py:56: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  path_box.on_submit(build_checks)
